# TrackLink USBL — Message vs. Log Timestamp Comparison

For each deployment that has both a parsed USBL message file and a parsed
TrackLink log fixes file, this notebook plots ship latitude, ship longitude,
target bearing, and target slant range as time series — one trace per source
— so the temporal relationship between the two can be inspected visually.

In [ ]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
MESSAGES_DIR: Path = Path("/data/exos_01/acfr_messages_v2_parsed")
LOGS_DIR: Path = Path("/data/exos_01/acfr_tracklink_logs_v2_parsed")

MSG_SUFFIX: str = "_usbl_tracklink.csv"
LOG_SUFFIX: str = "_tracklink_fixes.csv"

## Find matching deployment pairs

In [ ]:
msg_prefixes: set[str] = {
    path.name.removesuffix(MSG_SUFFIX)
    for path in MESSAGES_DIR.glob(f"*{MSG_SUFFIX}")
}
log_prefixes: set[str] = {
    path.name.removesuffix(LOG_SUFFIX)
    for path in LOGS_DIR.glob(f"*{LOG_SUFFIX}")
}

matched_prefixes: list[str] = sorted(msg_prefixes & log_prefixes)
print(f"Matched deployments: {len(matched_prefixes)}")
for prefix in matched_prefixes:
    print(f"  {prefix}")

## Load data

In [ ]:
MSG_COLS: list[str] = [
    "timestamp",
    "ship_latitude",
    "ship_longitude",
    "target_bearing_angle",
    "target_slant_range",
]
LOG_COLS: list[str] = [
    "timestamp",
    "ship_latitude",
    "ship_longitude",
    "target_bearing_angle",
    "target_slant_range",
]

pairs: list[tuple[str, pd.DataFrame, pd.DataFrame]] = []

for prefix in matched_prefixes:
    msg_path: Path = MESSAGES_DIR / f"{prefix}{MSG_SUFFIX}"
    log_path: Path = LOGS_DIR / f"{prefix}{LOG_SUFFIX}"

    msg: pd.DataFrame = pd.read_csv(
        msg_path,
        usecols=MSG_COLS,
        parse_dates=["timestamp"],
        date_format="ISO8601",
    )
    log: pd.DataFrame = pd.read_csv(
        log_path,
        usecols=LOG_COLS,
        parse_dates=["timestamp"],
        date_format="ISO8601",
    )

    pairs.append((prefix, msg, log))
    print(f"{prefix}: msg={len(msg)} rows, log={len(log)} rows")

print(f"\nLoaded {len(pairs)} / {len(matched_prefixes)} pairs")

## Timestamp offset: message vs. log

Matches entries across the two sources by rounding `target_bearing_angle` and
`target_slant_range` to two decimal places (to absorb float32 precision
differences) and inner-joining on those keys. For each matched pair the offset
is computed as `msg_timestamp − log_timestamp` in seconds.

All deployments are overlaid on a single plot.

In [ ]:
ROUND_DECIMALS: int = 2


def compute_offset(msg: pd.DataFrame, log: pd.DataFrame) -> pd.DataFrame:
    """Match message and log entries by rounded bearing+range, return timestamp offsets."""
    msg_keyed: pd.DataFrame = msg[["timestamp", "target_bearing_angle", "target_slant_range"]].copy()
    log_keyed: pd.DataFrame = log[["timestamp", "target_bearing_angle", "target_slant_range"]].copy()

    msg_keyed["bear_key"] = msg_keyed["target_bearing_angle"].round(ROUND_DECIMALS)
    msg_keyed["range_key"] = msg_keyed["target_slant_range"].round(ROUND_DECIMALS)
    log_keyed["bear_key"] = log_keyed["target_bearing_angle"].round(ROUND_DECIMALS)
    log_keyed["range_key"] = log_keyed["target_slant_range"].round(ROUND_DECIMALS)

    merged: pd.DataFrame = msg_keyed.merge(
        log_keyed,
        on=["bear_key", "range_key"],
        suffixes=("_msg", "_log"),
    )
    merged["offset_s"] = (
        merged["timestamp_msg"] - merged["timestamp_log"]
    ).dt.total_seconds()
    return merged[["timestamp_log", "offset_s"]].rename(columns={"timestamp_log": "timestamp"})


offsets_by_prefix: dict[str, pd.DataFrame] = {}

for prefix, msg, log in pairs:
    offsets: pd.DataFrame = compute_offset(msg, log)
    offsets_by_prefix[prefix] = offsets
    if offsets.empty:
        print(f"[no matches] {prefix}")
    else:
        print(
            f"{prefix}: {len(offsets)} matched pairs  "
            f"median offset = {offsets['offset_s'].median():.2f} s  "
            f"std = {offsets['offset_s'].std():.2f} s"
        )

In [ ]:
fig_offset = go.Figure()

for prefix, offsets in offsets_by_prefix.items():
    if offsets.empty:
        continue
    fig_offset.add_trace(
        go.Scatter(
            x=offsets["timestamp"],
            y=offsets["offset_s"],
            mode="markers",
            marker=dict(size=4, opacity=0.6),
            name=prefix,
        )
    )

fig_offset.update_layout(
    title="Timestamp offset: msg − log (seconds), all deployments",
    xaxis_title="Log timestamp",
    yaxis_title="Offset (s)",
    height=500,
    legend=dict(x=1.01, y=1.0, xanchor="left"),
    margin=dict(l=60, r=200, t=60, b=40),
)
fig_offset.show()

## Time-series comparison plots

One figure per deployment. Each figure has four rows:
- Ship latitude
- Ship longitude
- Target bearing angle (degrees)
- Target slant range (m)

Blue = USBL message source; orange = TrackLink log fixes source.

In [ ]:
CHANNELS: list[tuple[str, str, str]] = [
    ("ship_latitude", "Ship latitude (°)", "°"),
    ("ship_longitude", "Ship longitude (°)", "°"),
    ("target_bearing_angle", "Target bearing (°)", "°"),
    ("target_slant_range", "Target slant range (m)", "m"),
]

for prefix, msg, log in pairs:
    fig = make_subplots(
        rows=len(CHANNELS),
        cols=1,
        shared_xaxes=True,
        subplot_titles=[title for _, title, _ in CHANNELS],
        vertical_spacing=0.06,
    )

    for row, (col, title, unit) in enumerate(CHANNELS, start=1):
        show_legend: bool = row == 1

        fig.add_trace(
            go.Scatter(
                x=msg["timestamp"],
                y=msg[col],
                mode="lines+markers",
                marker=dict(size=3),
                line=dict(color="steelblue", width=1),
                name="Message",
                legendgroup="msg",
                showlegend=show_legend,
            ),
            row=row,
            col=1,
        )
        fig.add_trace(
            go.Scatter(
                x=log["timestamp"],
                y=log[col],
                mode="lines+markers",
                marker=dict(size=3),
                line=dict(color="darkorange", width=1),
                name="Log fixes",
                legendgroup="log",
                showlegend=show_legend,
            ),
            row=row,
            col=1,
        )
        fig.update_yaxes(title_text=unit, row=row, col=1)

    fig.update_layout(
        title=f"Message vs. log fixes — {prefix}",
        height=900,
        legend=dict(x=1.01, y=1.0, xanchor="left"),
        margin=dict(l=60, r=120, t=60, b=40),
    )
    fig.show()